# <span style="color:green"> Yummy Chummy Chatbot! - Food Ordering Chatbot.  </span>

## <span style="color:green"> CM2015 - Midterm Coursework  </span>

This is a food ordering chatbot that utilitizes regex notation to handle orders. It can show the menu, answers prices and general queries and can remember the user's name and current order.

### <span style="color:green"> Setting up the required Libraries </span>

This part is mainly concerned with setting of the required libraries:

- We'll import **"JSON"** so we can handle the "intents.json" file
- We'll import **"RE"** so we can handle regex notation
- We'll use **"RANDOM"** so we can select varied responses from the intents list
- Finally, we'll import the **"NLTK"** library so we can handle tokenisation, stopword removal and POS tagging

In [8]:
#  MIDTERM CODE

# ==================================================
# Yummy Chummy Chatbot - A Food Ordering Chatbot
# CM2015 - Programming With Data Midterm
# ==================================================

import json    # intents data file
import re      # regex pattern matching
import random  # varied response selection

import nltk  
from nltk.stem import PorterStemmer, WordNetLemmatizer  
from nltk.tokenize import word_tokenize                
from nltk.corpus import wordnet, stopwords              
from nltk.sentiment import SentimentIntensityAnalyzer  

# Downloading required NLTK datasets 
for pkg in ['punkt', 'punkt_tab', 'wordnet', 'omw-1.4',
            'averaged_perceptron_tagger_eng', 'stopwords', 'vader_lexicon',
            'maxent_ne_chunker', 'maxent_ne_chunker_tab', 'words']:
    
    try:    
        nltk.download(pkg, quiet=True)    
    except Exception:    
        pass         


<span style="font-family: 'Georgia'; font-size: 15px;">
    
## <span style="color:green"> Summary of Chatbot development: </span>

You will develop a data-driven **chatbot** in 4 stages. 
    
Each stage covers a core chatbot functionality - conversation loop, pattern matching, data-driven responses from files, and pre-processing with substitutions. 
    
This development will cover all learning materials from **weeks 1-10**. 

Please ensure that you learn respective weeks content before attempting different parts. Each stage will expand on previous stage by incorporating a new concept about data programming that you have learnt.

The following key concepts will be covered in each stage of chatbot development. 

- **Part 1.** Conversation loop
- **Part 2.** Pattern matching
- **Part 3.** File handling
- **Part 4.** NLTK and Word substitutions

<div style="text-align: center;">  <span style="color:red; font-size: 18px;"> <i> Please do these activities in a sequential order, starting from Part 1 to ending in Part 4. </i> </span> </div>



</span>


<hr style="border: 2px solid green;">

<span style="font-family: 'Georgia'; font-size: 15px;">

## <span style="color:green"> Chatbot : Part 1 </span>
    
**Conversation loop**

<!-- #### <span style="color:green"> 1. Chatbot with lists, conditions and string concatenation </span>
 -->

You will build the first part of the chatbot that responds to a fixed set of inputs using **lists** and **conditions**.

In order to complete this activity, please get yourself familiar with concepts covered from **Weeks 1 to Week 4** of the video material. 

**Learning outcomes**
- Creating a chatbot conversational loop with lists and conditonals
- String concatenation

Complete this activity before proceeding to the next parts of the chatbot.

</span>


---

Please review the following conversation. Your chatbot should demonstrate a behavior similar to this.


#### Sample conversation snippet:

> User: Hello

> Chatbot: Hello! How can I assist you today?

> User: I'd like to have some tea

> Chatbot: Sure, where would you like to have it? A cafe in the campus maybe?

> User: Yes, a cafe.

> Chatbot: There are 2 cafes - in main building and near the lecture halls.

> User: Thank you.

> Chatbot: You are welcome.

> User: Bye

-- Exits the chat --  

---

### <span style="color:green"> Main Loop </span>

In [9]:
# MIDTERM CODE

def chatbot():
    rgx2int, int2res, menu = load_files()     
    preprocessor = ChatbotPreprocessor()   
    user_state = {"name": "friend", "order": "order", "qty": "1", "basket": []}
    user_state["pricelist"] = "\n".join(f" {item}: ${p}" for item, p in menu.items())
    
    print("Yummy Chummy: Welcome to the Yummy Chummy Chatbot! Type 'exit' or 'quit' to leave.")
    
    # Exit only when 'exit' or 'quit' is typed
    while True:
        
        user_input = input("You: ")       
        
        if user_input.lower().strip() in ("exit", "quit"):
            print("Yummy Chummy: Goodbye!")   
            break                           
            
        # Matching order 
        tag = update_memory(user_input, user_state) 
        if tag is None:
            tag = match_intent(user_input, rgx2int) 
        if tag is None:            
            clean_input = preprocessor.process_input(user_input)   
            tag = match_intent(clean_input, rgx2int)       
            
        # item_inquiry is matched via the JSON pattern, but match intent doesn't update memory
        if tag == "item_inquiry":

            for text in (user_input, preprocessor.process_input(user_input)):
                item_match = re.search(r"\b(pizzas?|burgers?|pasta|salads?|fries|sodas?|coffee|teas?)\b", text, re.IGNORECASE)
                if item_match:
                    user_state["order"] = item_match.group(1).lower() 
                    break 
            
        # Sentinment fallback
        if tag is None and sentiment_analyzer:
            score = sentiment_analyzer.polarity_scores(user_input)["compound"]
            if score <= -0.5:   
                tag = "frustrated"    
        
        # Confirm the customer's order
        if tag == "confirm":
            if user_state["basket"]:
                lines = [f" {q} x {item} - ${menu[item] * q}" for q, item in user_state["basket"]]
                total = sum(menu[item] * q for q, item in user_state["basket"])
                user_state["summary"] = "\n".join(lines) + f"\n Total: ${total}"
                user_state["basket"].clear()
            else:
                tag = "checkout"
        
        # Give the customer a summary of their order
        if tag == "checkout":
            if user_state["basket"]:
                lines = [f" {q} x {item} - ${menu[item] * q}" for q, item in user_state["basket"]]
                total = sum(menu[item] * q for q, item in user_state["basket"])
                user_state["summary"] = "\n".join(lines) + f"\n Total: ${total}"
            else:
                user_state["summary"] = " (your basket is empty)"
        
        # Respones based on the tag
        if tag:
            print("Yummy Chummy: ", generate_response(tag, int2res, user_state))   
        else: 
            keywords = extract_keywords(user_input)    # keyword extraction makes the failure repsonse smarter
            if keywords: 
                print(f"Yummy Chummy: I didn't quite get that - is this about {', '.join(keywords)}?")  
            else: 
                print("Yummy Chummy: Sorry, could you repeat that again?")   


<div style="text-align: center;"> <span style="font-family: 'Georgia'; font-size: 25px; color:green"> End of part 1 </span> </div>
<hr style="border: 2px solid green;">


<span style="font-family: 'Georgia'; font-size: 15px;">

## <span style="color:green"> Chatbot : Part 2 </span>
**Pattern matching**


<!-- #### <span style="color:green"> 2. Chatbot with Dictionaries, and regular expressions </span> -->

You will enhance the basic chatbot to recognize patterns using **regular expressions** and organize code as **functions**.
        
Before proceeding, familiarise yourself with **Week 5 and 6** of lecture material on **Dictionaries** and **Regular expressions**.

<span style="color:red"> *Before you start this, please ensure that you complete all the previous parts (Part 1).* </span>


**Learning outcomes**
- Writing and storing patterns (2.1, 2.2). 
- Functions for pattern matching, and retrieval (2.3, 2.4). 
- String substitution (2.5, 2.6)

Complete this activity before proceeding to the next parts of the chatbot.

</span>


---

*<span style="color:red; font-size: 15px"> Modular programming: </span> We will be experimenting with several different types of pattern matching techniques for chatbot. As we do not want to end up with stray code for chatbot versions, for each new chatbot feature, we will write a function and call the functions in the main. At the end, we will have multiple functions for pattern matching and response generation, and we will be be in a position to mix and match them in the main loop for comparison.*

---

## Writing and storing patterns

You will write two functions to create dictionaries and store patterns in them. 

- First, you will use the ZIP function to combine two independent lists (precepts and Chatbot responses) 


- Then, you will create 5 regex patterns and store them in a dictionary named "re_patterns"


### 2.1 Lists -> Dictionaries

To convert lists to dictionaries, let's use the lists from previous activity.

Instead of writing a new *dictionary*, create this using the **ZIP** function on two lists. 

In [10]:
# NOT REQUIRED FOR MIDTERM - LAB PRACTICE ONLY

# # Write a function that takes in precept categories and responses lists and returns a mapping between them.

# # - Use ZIP function to combine two independent lists (precepts and responses) and create a new map between them.
# # - Search on the ZIP function and how to use it in a for loop
# # - Run the main loop and check that this function works

# # ___ Write your code here _____

# def zip_responses(precepts, response):
    
#     mapping = {}
    
#     for item_a, item_b in zip(precepts, response):
#         mapping[item_a] = item_b
        
#     return mapping


Use zip function to create a dictionary with 5 precepts-responses pairs. 

Use a list to accomodate more than one response. 

Example dictionary:
    
    {
        "greetings": ["how are you?"],
        "tea": ["I love this beverage", "Can you tell me more?, etc... "]
         ...
    }
    
   

In [11]:
# NOT REUQIRED FOR MIDTERM -LAB PRACTICE ONLY

# # Dictionary name : "responses" 

# # ___ Write your code here _____

# precepts = [
#     "greetings",
#     "tea",
#     "house",
#     "cafe",
#     "location"
# ]

# responses = [
#     ["how are you?", "what's up?", "how's it going?", "been a long time!", "how's your day?"],
#     ["I love this beverage", "tea is so delicious!", "you should definitely try some tea!", "drink tea when you need that boost in energy!", "try some tea with me sometime!"],
#     ["your house must be beautiful", "i can help you find houses nearby!", "if you need a house, i can help you!", "if you need a house to stay in, i can help!", "a house around here could be expensive, i can help you find it at a cheap rate!"],
#     ["there's a cafe nearby!", "I can show you the nearest cafe nearby!", "theres a cafe a few blocks away!", "i think you'll find a cafe interesting around here!", "Want a beverage? there's a cafe nearby!"],
#     ["we are currently in a nearby location!", "you could try out this location!", "this location is perfect", "this is a popular location", "this is such an amazing location!"]
# ]

# zip_responses(precepts, responses)

---

### 2.2 Regular expression -> Patterns -> Dictionary keys.

- Match complete words using regular expressions. 
- Example, match all occurrences of hello, hi, hey etc under a pattern called greetings.
- Use "|" symbol to string together multiple words.
- Each pattern should have atleast or 5 different occurrences of the word you want to match.
- Create atleast 5 different patterns.

Example dictionary:
    
    {"hi|hello|howdy|howru|yo" : "greetings", etc, etc }

---

In [12]:
# NOT INCLUDED IN MIDTERM

# # Create 5 RE patterns
# import re
# # Dictionary name: "re_patterns"

# # ___ Write your code here _____
# re_patterns = {
#     "hello|howdy|howru|yo|hey": "greetings", 
#     "tea|thirsty|beverage|drink|chai": "tea", 
#     "house|residence|place|area|home": "house", 
#     "cafe|coffee|doughnuts|pastry|hangout": "cafe", 
#     "location|surroundings|findings|site|spot": "location"
# }

## Functions: Pattern matching and response retrieval.

---

Using data created above, write programs to detect patterns and retrieve responses along for chatbot. 
 
<span style="color:red; font-size: 15px"> TIP: Here's a nice way to organize next bits of code. 
- Use a cell for each function (makes debugging easier)
- Auxillary functions to detect patterns and generate responses are in the cells above the main.
- The main python loop is the bottom most cell.
- Bespoke pattern matching code that you will write will be above these functions.

---

### 2.3 Function : Detect pattern from input

---

Use regular expressions + dictionary from before to detect a pattern from user input. 

**Function input** : User Input ("Hi, how are you?")  <br>
**Output**         : Response_type ("Greetings") 

---

In [13]:
# MIDTERM CODE

# Return the tag of the regex that matches, or don't return anything
def match_intent(user_input, rgx2int):
    
    for pattern, tag in rgx2int.items():
        if re.search(pattern, user_input, re.IGNORECASE):   
            return tag     
    return None     

# Capture personal information into user_state. Return the matching tag or return None
def update_memory(user_input, user_state):
    
    name_match = re.search(r"my name is (\w+)|i'?m called (\w+)", user_input, re.IGNORECASE)
    
    # Group 1 and 2 hold the captured word
    if name_match:
        user_state["name"] = (name_match.group(1) or name_match.group(2)).capitalize() 
        return "name_response"   
    
    # NER fallback for names to avoid false positives on other capitalized words
    if re.search(r"\bi'?m\b|\bi am\b|call me", user_input, re.IGNORECASE):
        entities = extract_entities(user_input)      
        if "PERSON" in entities:       
            user_state["name"] = entities["PERSON"].split()[0].capitalize()    
            return "name_response"  

    # Ordering phrases that require "I"
    order_match = re.search(
        r"(?:\bi(?:'d|'ll| would| will)? ?(?:like|want|order|take|have)\b"
        r"|(?:can|could) i (?:get|have|order)|give me)"
        r".*?\b(?:(\d+) )?(pizzas?|burgers?|pasta|salads?|fries|sodas?|coffee|teas?)\b",
        user_input, re.IGNORECASE)

    if order_match:
        item = order_match.group(2).lower()
        item = item if item in ("fries", "pasta") else item.rstrip('s')
        user_state["qty"] = order_match.group(1) or "1"  # \d+ quantity
        user_state["order"] = item
        user_state["basket"].append((int(user_state["qty"]), item))
        return "order_response"        
    return None        

### 2.4 Function : Retrieve response

---

Use the output of pattern detection function (above) to trigger a chatbot response. 

In the function, use the "responses" dictionary from above. 


Follow the steps to write function. The function returns a response (e.g., "hello", or "how are you").

**Function input** : User Input ("Hi, how are you?")  <br>
**Detect pattern** : Response_type ("Greetings")  <br>
**Output**         : One of ["hello", "how are you?", etc...]

---

In [14]:
# MIDTERM CODE

def generate_response(tag, int2res, user_state):
    
    # Random template for the tag
    if tag in int2res:
        template = random.choice(int2res[tag])     
        return template.format(**user_state)      
    return "I'm sorry, could you repeat that?"     

---

<span style="font-family: 'Georgia'; font-size: 15px;">

<span style="color:red;">  Before you proceed .... </span>

- Save the notebook and make a duplicate copy of this notebook 
- Name that copy - chatbot_re_basic_version 
- Now, we will proceed to make more changes to other parts of chatbot code. 

<span>

---

## String substitution


Variable parts can be introduced in a string and substituted with meaningful patterns from user input to make the chatbot's responses more specific. 

This is achieved by string substituitions. Typical substituions can be about user's name, their favourite color, something they mention in the earlier part of the conversation.

Let's look at following example in context of a question & answer chatbot.

### 2.5 Pre-defined templates and responses  <br>

<span style="font-family: 'Georgia'; font-size: 15px;">

The lectures gave us a great starting point, but we're going to start to produce our own independent work now.

In this section, you will further customize chatbot with pre-defined template repsonses and advanced regular expressions.

**Only attempt this after finishing the basic chatbot.**

---

In [15]:
# NT REQUIRED FOR MIDTERM - CHATBOT LAB PRACTICE

# print('I have a {food_item} and a {drink_item} with me'.format(drink_item='soda', food_item='sandwich'))

# # 1. Define necessary variables and complete the following print statement. Uncomment and run it. 

# print('We have {number} {container} containing {qty} gallons of {food_item}'.format(number="3", container="cans", qty="50", food_item="pepsi"))

# # 2. Now, store all the variables in a dictionary. Use the dictionary to complete the following print statement. Uncomment and run it. 

# variables = {
#     "number": "3",
#     "container": "cans",
#     "qty": "50",
#     "food_item": "pepsi"
# }

# print('We have {number} {container} containing {qty} gallons of {food_item}'.format(**variables))


Now, rewrite some of your older responses as template strings and see if you can substitue them with values from user input. 

If you want to attempt a newer one for user name and favourite color, follow the instructions below.

---

In [16]:
# NOT REQUIRED FOR MIDTERM - CHATBOT LAB PRACTICE 

# # ___ Write your code here _____
# feedback_template = "Well done {student}!, You built most of the {project}. We wish further sucess to you!"

# name = input("What is your name? ")
# project_name = input("What project were you working on? ")

# print(feedback_template.format(student=name, project=project_name))

### 2.6 Advanced regular expressions

Let's exploring advanced REs by creating two more patterns to match the name and favourite color of the user. 

As name, and color can be any value, pattern matching needs to be generic and requires using alphabets [a-zA-Z], words [\w] and special symbols [*,+,?].

In [17]:
# Write a regular expression pattern for a name and favourite color.

# 1. Names typically start with a letter in upper case followed by a sequence of letters in lower case.
# Try writing a few use cases to ensure that the pattern works.

# 2. Colors typically start with a pattern - color is <color_name> or I like <color_name>.
# Try writing a few use cases to ensure that the pattern works.


In [18]:
# NOT REQUIRED FOR MIDTERM - CHATBOT LAB PRACTICE 

# # ___ Write your code here _____
# re_patterns = { "name": r"my name is ([A-Z][a-z]+)", 
#                 "color": r"(?:color is|I like) (\w)"
#               }

# # For 1 and 2
# test_cases = [
#     "My name is Wiqar",
#     "My favourite color is black",
#     "stacey likes the same color too",
#     "i like black as an outfit color"
# ]

# for case in test_cases:
#     print(f"\nTesting: '{case}'")
#     for category, pattern in re_patterns.items():
#         match = re.search(pattern, case, re.IGNORECASE)
#         if match:
#             print(f"Correct, You found {category}: {match.group(1)}")
            

# test_strings = [
#     "Hello, I am Sarah and color is red",
#     "i like green shoes",
#     "David doesn't care about colors"
# ]
            
# # Write few examples of input strings containing names and color to ensure that the pattern works.
# for text in test_strings:
#     for key, pattern in re_patterns.items():
#         match = re.search(pattern, text)
#         if match:
#             print(f" -> Match found for {key}: {match.group()}")
#     print("-" * 30)

#### Integrating name and color into the chatbot 

Lastly, let's integrate the new features into the chatbot.

<span style="color:red; font-size: 15px; position:center "> Be very methodological as you start integrating these new patterns! At every step, pause, write a test case, check for outputs and then proceed! </span>  


##### 1. Expanding pattern detection (detect_pattern function)
    - Create user_state as a global variable dictionary above the detect_pattern function.
    - Along with other if-else conditions, add two regular expression patterns for finding name and favourite color.
    - If there is a match with name or color
        - Get the result of the match using result.group()
        - Store the match in the corresponding key of the user_state dictionary.
        - return "name_response" or "color_response" according to the condition

##### 2. Expanding responses
    - Add the two responses for the new categories - name and color. 
    - These responses should be in the format - 'hello {name}, how are you today?' or 'Great to know that your favourite color is {color}'

##### 3. Modifying chatbot responses
    - In chatbot_response, following the function call to detect_pattern, create 2 variables, name and color.
    - Store the values from user_state in name and color.
    - Along with other if-else conditions, add two conditions for name_response and color_response
    - If there is a match,
        - Retrieve the match from responses dictionary and store it as "response"
        - Use string substitution operation (response.format) with the name or color variable.

#### At this point, verify that the code works correctly!!


<div style="text-align: center;"> <span style="font-family: 'Georgia'; font-size: 25px; color:green"> End of part 2 </span> </div>
<hr style="border: 2px solid green;">

<span style="font-family: 'Georgia'; font-size: 15px;">

## <span style="color:green"> Chatbot : Part 3 </span>

**File handling**
    
In this part, you will create and load a JSON file to drive the chatbot's responses. At the end of this step,
we will be truly close to achieving full independence between data and process/program. 
 
Pre-readings: Lecture material of **Week 7 and 8** on **File handling**. Please read before before attempting.
    
<span style="color:red"> *Before you start, complete all the previous parts (Parts 1 and 2).*

**Learning outcomes**
- Reading data from JSON file
- Populating chatbot knowledge structures (patterns,intents,responses)
- Integrating file-based data into chatbot logic

Complete this activity before proceeding to the next parts of the chatbot.

</span>

---

### 3.1 Reading chatbot intents from a JSON file

Apply the file handling concepts and pandas to read JSON file (intents.json) and store its content.


In [19]:
# NOT REQUIRED FOR MIDTERM - CHATBOT LAB PRACTICE

# ### Write the function - read_json()...
# import pandas as pd
# import json

# def read_json():

#     with open('intents.json', 'r') as f:
#         data = json.load(f)
    
#     return pd.DataFrame(data['intents'])

In [20]:
# NOT REQUIRED FOR MIDTERM - CHATBOT LAB PRACTICE

# ### Call the function and store intents in intents_json

# ## Write code here...
# intents_json = read_json()

In [21]:
# NOT REQUIRED FOR MIDTERM - CHATBOT LAB PRACTICE

# ### Print print only the patterns of intents_json to see 
# ### their content as that's directly relevant for next step.
# print(intents_json['patterns'])


--- 

**Here are some questions to help you guide your decision-making for next steps**:

- From some of the intent/response pairs , what do the contents seem to be about? 
- Are all patterns strings? Are some regular expressions? 
- What is the best format to convert them to? 

--- 

---

### 3.2 Populating chatbot knowledge structures (patterns,intents,responses)

The logical choice for creating these mappings is to use dictionaries. 

**Remember**: While constructing a dictionary, you want the the **key-value** to be same data type. But, patterns are regular expressions, intents are strings, and responses are lists or strings. 

In the following cells, practice type conversions from strings to regular expressions, to construct the right dictionaries for mappings. 

Pay attention to the different formats while constructing these dictionaries.

---

### Strings -> regex

In [22]:
# NOT REQUIRED FOR MIDTERM - CHATBOT LAB PRACTICE

# # To convert patterns from strings to REs, we have to look 
# # at each pattern and concatenate them with a pipe symbol.

# # Loop thorugh patterns list and tags.
#  # For each pattern in list, add an "r" symbol
#  # Create a dictionary with pattern and tags

# # Create an empty dictionary mapping pattern to tags
# rgx2int = {}

# for index, row in intents_json.iterrows():
    
#     regex_string = r"|".join(row['patterns'])

#     rgx2int[regex_string] = row['tag']

# print(rgx2int)

### Dictionary 1: Patterns -> intent 

In [23]:
#  NOT REQUIRED FOR MIDTERM - CHATBOT LAB PRACTCIE

# ## Create an empty dictionary variable rgx2int for mapping from pattern to tags/intents

# ## Write code here....
# rgx2int = {}

# for index, row in intents_json.iterrows():
#     regex_string = "|".join(row['patterns'])
#     rgx2int[regex_string] = row['tag']
# print(rgx2int)

### Dictionary 2: intent -> responses


TIP: For any feature, write this as a function. (e.g. store_responses)

In [24]:
# NOT REQUIRED FOR MIDTERM - CHATBOT LAB PRACTICE

# ## Create an empty dictionary int2res for mapping from intent to responses.

# # Loop through the list of JSON objects:
#  # For each object in list, get responses and tag
#  # Add to the dictionary: tags as keys and responses as values
# int2res = {}

# for index, row in intents_json.iterrows():
#     int2res[row['tag']] = row['responses']
# print(int2res)

### 3.3 Integrating file-based data into chatbot logic

Follow the code templates provided below to understand how files can be integrated with the Chatbot logic. 

*Fill in appropriate code segments.*


In [25]:
# MIDTERM CODE

# The Load Files function
def load_files(path='intents.json'):
    """Load the intents JSON file and then build patterns"""
    
    with open(path, 'r') as file:   
        data = json.load(file)      
        
    rgx2int = {}       
    int2res = {}       
    
    for intent in data['intents']:
        # Each intent has patterns that need to be joined
        regex_string = "|".join(intent['patterns'])      
        rgx2int[regex_string] = intent['tag']            
        int2res[intent['tag']] = intent['responses']     
    return rgx2int, int2res, data['menu']   


<div style="text-align: center;"> <span style="font-family: 'Georgia'; font-size: 25px; color:green"> End of part 3 </span> </div>
<hr style="border: 2px solid green;">

<span style="font-family: 'Georgia'; font-size: 15px;">

## <span style="color:green"> Chatbot : Part 4 </span>        
**NLTK and Word substitutions**
    
<span style="color:red; font-size: 15px">**Note:**</span> *You will notice that this part is
bit less structured than previous parts. As you proceed to finalise 
the details of your chatbot for submission, 
reflect and understand the
code structure from the previous parts. Use this observation to 
meaningfully extend the chatbot
new pre-processing features.*



Before matching patterns, the chatbot should map certain words to alternatives that are easier for the chatbot to handle.

In this part, you will implement stemmers and lemmatizers that perform these substitutions on the user’s input before further processing.
    
Please get yourself familiar with **Week 9 and 10** of lecture material on **NLTK and text pre-processing** before attempting this. 

<span style="color:red"> *Before you start this, please ensure that you complete all the previous parts (Parts 1, 2 and 3).* </span>

**Learning outcomes**
- Tokenizing and preprocessing user input (punctuation, and stopwords)
- Word substitutitons (stemming, and lemmatization)
- Integrating preprocessing into chatbot pipeline

This is the last part of the chatbot. 

</span>

--- 


### Word substitutions

Before matching patterns, the chatbot should map certain words to alternatives that are easier for the chatbot to handle. For example:

    discussed → discuss 
    running → run 


<span style="font-family: 'Georgia'; font-size: 18px; "> Identify stems and root words </span>
    
In the following conversational snippet, identify how stemmers and lemmatizers are used to identify word variations.

-> Stemmer is used to identify "ing" forms of a verb (e.g., running and run) <br>
-> Lemmatizer is used recognize words with same root word (e.g., better and good)

    
The chatbot you build should demonstrate a behavior similar to this.

### Sample conversation snippet:

> User: I was running late to the NLP class today.

> Chatbot: Oh no! Were you feeling stressed?

> User: Yes, I was worrying about missing the lecture on stemmers and lemmatizers.

> Chatbot: Don’t worry, they were just discussing the stemmers and lemmatizers.

> User: Can you explain what they discussed?

> Chatbot: They explained how stemmers just cut off word endings, while lemmatizers consider the context.

> User: Oh, so like running becomes run? 

> Chatbot: Yes, running becomes run (**Stemmer applied**)

> User: What does better become?

> Chatbot: "Running" becomes "run" in both, but “better” becomes “good” only in lemmatization (**Lemmatizer applied**)

> User: That makes sense. So, choosing between them depends on the context!

> Chatbot: Yes. If speed is important, use stemming. If accuracy matters, use lemmatization.

> User: Thanks

> Chatbot: No worries! Have a great day!


In [26]:
# ChatbotPreProcessor Class
class ChatbotPreprocessor:
    
    def __init__(self): 
        
        self.stemmer = PorterStemmer()     
        self.lemmatizer = WordNetLemmatizer()  
        
        # Fallback to any built in stopwords
        try:  
            self.stop_words = set(stopwords.words('english')) 
        except LookupError: 
            self.stop_words = {"a", "an", "the", "is", "are", "am", "i",
                               "you", "to", "of", "and", "or", "in", "on",
                               "for", "it","this", "that", "my", "your"} 
            
        # Normalizing the food vocabulary
        self.synonyms = {
            "starving": "hungry",
            "famished": "hungry",
            "peckish": "hungry",
            "coke": "soda",
            "pop": "soda",
            "chips": "fries"
        }
        

    # Map NLTK POS tags to WordNet POS tags for proper lemmatization
    def _get_wordnet_pos(self, word):
        
        tag = nltk.pos_tag([word])[0][1][0].upper()   
        
        tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN,
                    "V": wordnet.VERB, "R": wordnet.ADV}   
        
        return tag_dict.get(tag, wordnet.NOUN)     
    
    # Replace words with their synonyms
    def handle_synonyms(self, tokens):
        return [self.synonyms.get(word, word) for word in tokens] 
    
    # Chop word endings
    def apply_stemming(self, tokens):
        return [self.stemmer.stem(word) for word in tokens]   
    
    # Reduce words to their dictionary roots
    def apply_lemmatization(self, tokens):
        return [self.lemmatizer.lemmatize(w, self._get_wordnet_pos(w)) for w in tokens] 
    
    # Full pipeline, falls bakc gracefully
    def process_input(self, text, strategy='lemmatize'):
        
        try:
            tokens = word_tokenize(text.lower())   
        except LookupError:   
            tokens = re.findall(r"\w+", text.lower())     # fallback tokenizer
            
        tokens = [w for w in tokens if w.isalnum()]     
        tokens = [w for w in tokens if w not in self.stop_words] 
        tokens = self.handle_synonyms(tokens) 
        
        try:  
            if strategy == 'stem':   
                tokens = self.apply_stemming(tokens)    
            elif strategy == 'lemmatize':              
                tokens = self.apply_lemmatization(tokens) 
        except LookupError:    
                pass         # keep unlemmatized tokens
        return " ".join(tokens)  


## Further steps:
    
### Synonym Handling 

Your chatbot can be further improved by handling synonyms through substitution. 
    
Create a rule file with sets of synonyms that the chatbot should treat as equivalent for 
pattern matching purposes. For example, the words "sad", "unhappy", and "depressed" are considered synonymous.

You will need to implement synonym substitution, where the input is normalized before attempting to match any   patterns. For example, if a user says, "I am unhappy," the chatbot should recognize that "unhappy" is  
synonymous with "sad."



<div style="text-align: center;"> <span style="font-family: 'Georgia'; font-size: 25px; color:green"> End of part 4 </span> </div>
<hr style="border: 2px solid green;">

### Sentiment Analyzer

In [27]:
# VADER Detetcs negative messages so the bot can apologize
try:
    sentiment_analyzer = SentimentIntensityAnalyzer()  
except LookupError: 
    sentiment_analyzer = None     # Bot will work but this feature would be excluded then

### Extract Keywords

In [28]:
# Extract any key words 
def extract_keywords(text, top_n=3):
    
    tokens = preprocessor.process_input(text).split()
    
    try: 
        tagged = nltk.pos_tag(tokens)  
        keywords = [w for w, t in tagged if t.startswith(('NN', 'VB', 'JJ'))]     
    except LookupError:   
        keywords = tokens         
    return keywords[:top_n]      

### Extract Entities

In [29]:
# Extract the entities from the text
def extract_entities(text):
    
    try:
        tree = nltk.ne_chunk(nltk.pos_tag(word_tokenize(text))) 
    except LookupError:    
        return {}    
    entities = {}    
    
    for subtree in tree:  
        if hasattr(subtree, 'label'):         
            entities[subtree.label()] = " ".join(w for w, t in subtree.leaves())
    return entities 

### Test Cases

Here are a few test cases that I wrote that would demonstrate the chatbot's functinality. 

In [30]:
# Capture the first 3 but don't capture the last
memory_tests = [
    "My name is Wiqar",
    "I'd like some pizza",
    "can i get some pizza?",
    "do you have pizza?"
] 

user_state = {"name": "friend", "order": "order", "qty": "1", "basket": []}    

for case in memory_tests:
    tag = update_memory(case, user_state)     
    print(f"{case!r} -> {tag}, memory = {user_state}")  
    
# 2nd Test, Check for End-To-End detection
rgx2int, int2res, menu = load_files()          
preprocessor = ChatbotPreprocessor()    
user_state = {"name": "friend", "order": "order", "qty": "1", "basket": []}    

# A complete dictionary on the intent test cases
intent_tests = [
    ("Hello there!", "greeting"),
    ("My name is Wiqar", "name_response"),
    ("What's on the menu?", "menu"),
    ("I'll take on a burger", "order_response"),
    ("how much is a pizza?", "price"),
    ("do you deliver?", "delivery"),
    ("I am absolutely famished", "hungry"),       # synonym
    ("cancel that", "cancel"),
    ("hajdhkjdabhadbhkghehkjgaekhd", None),         # This shouldn't crash the bot
    ("this is terrible service!", "frustrated")
]

for text, expected in intent_tests:

    tag = update_memory(text, user_state) or match_intent(text, rgx2int)
    if tag is None:
        tag = match_intent(preprocessor.process_input(text), rgx2int)
    status = "PASS" if tag == expected  else "FAIL"      
    print(f"{status}: {text!r} -> {tag}")    

'My name is Wiqar' -> name_response, memory = {'name': 'Wiqar', 'order': 'order', 'qty': '1', 'basket': []}
"I'd like some pizza" -> order_response, memory = {'name': 'Wiqar', 'order': 'pizza', 'qty': '1', 'basket': [(1, 'pizza')]}
'can i get some pizza?' -> order_response, memory = {'name': 'Wiqar', 'order': 'pizza', 'qty': '1', 'basket': [(1, 'pizza'), (1, 'pizza')]}
'do you have pizza?' -> None, memory = {'name': 'Wiqar', 'order': 'pizza', 'qty': '1', 'basket': [(1, 'pizza'), (1, 'pizza')]}
PASS: 'Hello there!' -> greeting
PASS: 'My name is Wiqar' -> name_response
PASS: "What's on the menu?" -> menu
PASS: "I'll take on a burger" -> order_response
PASS: 'how much is a pizza?' -> price
PASS: 'do you deliver?' -> delivery
PASS: 'I am absolutely famished' -> hungry
PASS: 'cancel that' -> cancel
PASS: 'hajdhkjdabhadbhkghehkjgaekhd' -> None
PASS: 'this is terrible service!' -> frustrated


In [ ]:
# Call the Yummy Chummy ChatBot in the final Cell after all the code has been executed
chatbot()

Yummy Chummy: Welcome to the Yummy Chummy Chatbot! Type 'exit' or 'quit' to leave.
You: hi
Yummy Chummy:  Hi there friend, hungry? I can take your order!
You: Yes what do you have?
Yummy Chummy:  We serve pizza, burgers, salads, pasta and fries, plus soda, coffee and tea to drink.
You: I'll take a pizza
Yummy Chummy:  One pizza coming right up, friend!
You: I'll take 2 more pizzas
Yummy Chummy:  One pizza coming right up, friend!
You: I'll take 2 sodas
Yummy Chummy:  Great choice! Adding a soda to your order.
You: Add a soda
Yummy Chummy:  soda is a great choice - just tell me you want it and its yours!
You: I want a soda
Yummy Chummy:  A soda it is! Any beverage you'd want with that?
You: Alright thats it
Yummy Chummy:  Here's your order, friend:
 1 x pizza - $8
 1 x pizza - $8
 2 x soda - $4
 1 x soda - $2
 Total: $22
Say 'confirm' to place it!
You: confirm
Yummy Chummy:  Confirmed, friend! See you soon.
